# 01 · Who is a player? Team clustering on the test clip

**Goal:** the smoke test found a median of 17 people per frame. Separate them into team A, team B, referees and everyone else (bench, fans, staff) without any labels, so tracking only follows the 10 players.

**Method:** detect people with RF-DETR → crop the torso of each box → embed crops with SigLIP → reduce with UMAP → k-means with k = 4 → judge clusters by contact sheets. A second signal: whether the box's foot point lies on the floor area (bottom-centre below a horizon line estimated from the frame).

**Runtime:** L4 GPU, about 15 minutes.

**Pass criteria:** two clusters that each look like one team's jersey, one cluster of referees, one of others; per-frame count of the two team clusters close to 10.

**Outputs:** `results/01_team_clustering/` → `metrics.json`, `cluster_0.png` … `cluster_3.png` (contact sheets of 48 crops each), `counts_per_frame.csv`.

In [ ]:
import os, json, time, subprocess
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/MSU-Alpha-3'
REPO = '/content/MSU-alpha-tree'
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/aldemirkonuk/MSU-alpha-tree.git', REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull', '-q'], check=True)
subprocess.run(['pip', 'install', '-q', 'rfdetr', 'supervision', 'umap-learn', 'transformers', 'scikit-learn'], check=True)
OUT = f'{REPO}/results/01_team_clustering'; os.makedirs(OUT, exist_ok=True)
CLIP = f'{DRIVE}/raw_video/test/playon_sample_120s.mp4'
assert os.path.exists(CLIP), 'Run notebook 00 first'


In [ ]:
# Detect people on 2 frames per second, keep boxes and torso crops
import cv2, numpy as np
from PIL import Image
from rfdetr import RFDETRBase
try:
    from rfdetr.util.coco_classes import COCO_CLASSES
    PERSON = next(k for k, v in COCO_CLASSES.items() if v == 'person')
except Exception:
    PERSON = 1
model = RFDETRBase()
cap = cv2.VideoCapture(CLIP); fps = cap.get(cv2.CAP_PROP_FPS); n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)); W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
step = max(int(round(fps / 2)), 1)
crops, meta = [], []
for fi in range(0, n, step):
    cap.set(cv2.CAP_PROP_POS_FRAMES, fi); ok, frame = cap.read()
    if not ok: break
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    det = model.predict(Image.fromarray(rgb), threshold=0.4)
    for (x1, y1, x2, y2), c, s in zip(det.xyxy, det.class_id, det.confidence):
        if c != PERSON: continue
        x1, y1, x2, y2 = map(int, (x1, y1, x2, y2)); bh = y2 - y1
        if bh < 40: continue
        ty1, ty2 = y1 + int(0.15 * bh), y1 + int(0.55 * bh)   # torso band: jersey
        crop = rgb[max(ty1,0):min(ty2,H), max(x1,0):min(x2,W)]
        if crop.size == 0: continue
        crops.append(Image.fromarray(crop).resize((96, 96)))
        meta.append({'frame': fi, 'x1': x1, 'y1': y1, 'x2': x2, 'y2': y2, 'conf': float(s), 'foot_y': y2, 'box_h': bh})
print(len(crops), 'person crops from', len(set(m['frame'] for m in meta)), 'frames')


In [ ]:
# Embed with SigLIP, reduce with UMAP, cluster with k-means (k = 4)
import torch, umap
from transformers import AutoProcessor, SiglipVisionModel
from sklearn.cluster import KMeans
dev = 'cuda'
proc = AutoProcessor.from_pretrained('google/siglip-base-patch16-224')
enc = SiglipVisionModel.from_pretrained('google/siglip-base-patch16-224').to(dev).eval()
embs = []
with torch.no_grad():
    for i in range(0, len(crops), 64):
        batch = proc(images=crops[i:i+64], return_tensors='pt').to(dev)
        embs.append(enc(**batch).pooler_output.cpu().numpy())
E = np.concatenate(embs)
Z = umap.UMAP(n_components=3, random_state=0).fit_transform(E)
K = 4
labels = KMeans(n_clusters=K, n_init=10, random_state=0).fit_predict(Z)
for m, l in zip(meta, labels): m['cluster'] = int(l)
print({k: int((labels == k).sum()) for k in range(K)})


In [ ]:
# Contact sheets: 48 random crops per cluster, for human / planner review
import random
random.seed(0)
for k in range(K):
    idx = [i for i, l in enumerate(labels) if l == k]
    pick = random.sample(idx, min(48, len(idx)))
    sheet = Image.new('RGB', (8 * 96, 6 * 96), 'white')
    for j, i in enumerate(pick):
        sheet.paste(crops[i], ((j % 8) * 96, (j // 8) * 96))
    sheet.save(f'{OUT}/cluster_{k}.png')
print(sorted(os.listdir(OUT)))


In [ ]:
# Per-frame counts by cluster; which two clusters behave like teams (about 5 each per frame)
import pandas as pd
df = pd.DataFrame(meta)
per = df.groupby(['frame', 'cluster']).size().unstack(fill_value=0)
per.to_csv(f'{OUT}/counts_per_frame.csv')
summary = {int(k): {'total_crops': int((labels == k).sum()),
                    'median_per_frame': float(per[k].median()) if k in per else 0.0,
                    'median_box_height_px': float(df[df.cluster == k].box_h.median()),
                    'median_foot_y_px': float(df[df.cluster == k].foot_y.median())} for k in range(K)}
team_guess = sorted(summary, key=lambda k: abs(summary[k]['median_per_frame'] - 5))[:2]
players_per_frame = per[team_guess].sum(axis=1) if all(t in per for t in team_guess) else per.sum(axis=1)*0
metrics = {'notebook': '01_team_clustering', 'ran_at_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
           'crops': len(crops), 'frames': int(per.shape[0]), 'k': K, 'clusters': summary,
           'team_clusters_guess': [int(t) for t in team_guess],
           'players_per_frame_from_team_clusters': {'median': float(players_per_frame.median()), 'p10': float(players_per_frame.quantile(0.1)), 'p90': float(players_per_frame.quantile(0.9))}}
with open(f'{OUT}/metrics.json', 'w') as f: json.dump(metrics, f, indent=2)
os.makedirs(f'{DRIVE}/reports/01_team_clustering', exist_ok=True)
subprocess.run(['cp', '-r', OUT + '/.', f'{DRIVE}/reports/01_team_clustering/'])
print(json.dumps(metrics, indent=2))


**After the run (runner):** commit `results/01_team_clustering/` (metrics, the 4 contact sheets, counts CSV). Describe in the task file what each contact sheet shows (e.g. "cluster 2: white jerseys").